In [1]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0"

In [2]:
# Install Java 17 (Required for Spark)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

Get:1 https://cli.github.com/packages stable InRelease [3917 B]
Hit:2 https://packages.cloud.google.com/apt cloud-sdk InRelease                
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:8 https://archive.ubuntu.com/ubuntu noble InRelease                        
Hit:9 https://security.ubuntu.com/ubuntu noble-security InRelease   
Hit:10 https://archive.ubuntu.com/ubuntu noble-updates InRelease    
Hit:11 https://cloud.archive.ubuntu.com/ubuntu noble InRelease
Hit:12 https://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:13 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:14 https://cloud.archive.ubuntu.c

In [3]:
import sqlite3
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Set JAVA_HOME and initialize Spark Session with specific configurations
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


spark = SparkSession.builder \
        .master("local[4]") \
        .appName("PySpark DataFrames API") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

print("Spark Session configured and ready!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 18:00:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session configured and ready!


In [7]:
df = spark.read.csv('../ds1/Messy_Employee_dataset.csv', header=True, inferSchema=True)

In [34]:
analysis_index_dupli = df

In [8]:
messy = df

In [24]:
#set employee_id as index
messy = messy.withColumn("employee_id", col("employee_id").cast("string"))

In [9]:
messy.show(15)

+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|Employee_ID|First_Name|Last_Name| Age|  Department_Region|  Status| Join_Date|   Salary|               Email|      Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+----+-------------------+--------+----------+---------+--------------------+-----------+-----------------+-----------+
|    EMP1000|       Bob|    Davis|  25|  DevOps-California|  Active|  4/2/2021| 59767.65|bob.davis@example...|-1651623197|          Average|       true|
|    EMP1001|       Bob|    Brown|NULL|      Finance-Texas|  Active| 7/10/2020| 65304.66|bob.brown@example...|-1898471390|        Excellent|       true|
|    EMP1002|     Alice|    Jones|NULL|       Admin-Nevada| Pending| 12/7/2023|  88145.9|alice.jones@examp...|-5596363211|             Good|       true|
|    EMP1003|       Eva|    Davis|  25|       Admin-Nevada|Inactive|11/27/2021| 69

In [14]:
for col_name, dtype in df.dtypes:
    print(f"{col_name}: {dtype}")

Employee_ID: string
First_Name: string
Last_Name: string
Age: int
Department_Region: string
Status: string
Join_Date: string
Salary: string
Email: string
Phone: bigint
Performance_Score: string
Remote_Work: boolean


salary is string

In [12]:
numeric_cols = [
    col_name
    for col_name, dtype in messy.dtypes
    if dtype in ['int', 'bigint', 'double', 'float', 'long', 'decimal']
]

In [16]:
messy.describe(["Age", "Salary"]).show()

+-------+-----------------+------------------+
|summary|              Age|            Salary|
+-------+-----------------+------------------+
|  count|              809|              1020|
|   mean|32.48454882571075| 85155.05639558229|
| stddev|5.656860469525906|19873.727918135464|
|    min|               25|         100123.15|
|    max|               40|               N/A|
+-------+-----------------+------------------+



In [20]:
df.summary().show()

+-------+-----------+----------+---------+-----------------+-----------------+-------+---------+------------------+--------------------+--------------------+-----------------+
|summary|Employee_ID|First_Name|Last_Name|              Age|Department_Region| Status|Join_Date|            Salary|               Email|               Phone|Performance_Score|
+-------+-----------+----------+---------+-----------------+-----------------+-------+---------+------------------+--------------------+--------------------+-----------------+
|  count|       1020|      1020|     1020|              809|             1020|   1020|     1020|              1020|                1020|                1020|             1020|
|   mean|       NULL|      NULL|     NULL|32.48454882571075|             NULL|   NULL|     NULL| 85155.05639558229|                NULL|-4.942252959221569E9|             NULL|
| stddev|       NULL|      NULL|     NULL|5.656860469525906|             NULL|   NULL|     NULL|19873.727918135464|     

In [18]:
n_rows = messy.count()
n_cols = len(messy.columns)

print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")

Rows: 1020
Columns: 12


In [19]:
from pyspark.sql import functions as F

messy.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in messy.columns
]).show()

+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+
|Employee_ID|First_Name|Last_Name|Age|Department_Region|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+
|          0|         0|        0|211|                0|     0|        0|     0|    0|    0|                0|          0|
+-----------+----------+---------+---+-----------------+------+---------+------+-----+-----+-----------------+-----------+



In [21]:
missing_pct = messy.select([
    (F.count(F.when(F.col(c).isNull(), c)) / F.count("*") * 100).alias(c)
    for c in messy.columns
])

missing_pct.show(truncate=False)

+-----------+----------+---------+-----------------+-----------------+------+---------+------+-----+-----+-----------------+-----------+
|Employee_ID|First_Name|Last_Name|Age              |Department_Region|Status|Join_Date|Salary|Email|Phone|Performance_Score|Remote_Work|
+-----------+----------+---------+-----------------+-----------------+------+---------+------+-----+-----+-----------------+-----------+
|0.0        |0.0       |0.0      |20.68627450980392|0.0              |0.0   |0.0      |0.0   |0.0  |0.0  |0.0              |0.0        |
+-----------+----------+---------+-----------------+-----------------+------+---------+------+-----+-----+-----------------+-----------+



Change NULL strings to null values so they are detected in null count

In [25]:
total_rows = messy.count()
unique_rows = messy.dropDuplicates().count()

print("Duplicated rows:", total_rows - unique_rows)

Duplicated rows: 0


In [31]:
messy.groupBy("First_Name", "Last_Name", "Age", "Email") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+----------+---------+----+--------------------+-----+
|First_Name|Last_Name| Age|               Email|count|
+----------+---------+----+--------------------+-----+
|     Heidi|    Smith|  25|heidi.smith@examp...|    6|
|     Alice|   Miller|  25|alice.miller@exam...|    3|
|     Alice|    Brown|  40|alice.brown@examp...|    4|
|     Heidi|   Garcia|  40|heidi.garcia@exam...|    5|
|     Heidi|    Jones|  35|heidi.jones@examp...|    2|
|     Frank|  Johnson|  25|frank.johnson@exa...|    4|
|       Eva|    Brown|  35|eva.brown@example...|    3|
|       Bob|   Garcia|  30|bob.garcia@exampl...|    4|
|     David|   Miller|  30|david.miller@exam...|    2|
|     Grace|    Smith|  40|grace.smith@examp...|    4|
|   Charlie|    Davis|  35|charlie.davis@exa...|    3|
|     Frank|    Davis|  25|frank.davis@examp...|    2|
|       Eva|    Brown|NULL|eva.brown@example...|    4|
|     Alice|  Johnson|  40|alice.johnson@exa...|    3|
|     David|    Davis|  35|david.davis@examp...|    2|
|     Davi

In [33]:
messy.groupBy("First_Name", "Last_Name", "Age", "Phone") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+----------+---------+---+-----+-----+
|First_Name|Last_Name|Age|Phone|count|
+----------+---------+---+-----+-----+
+----------+---------+---+-----+-----+



Check unique values

In [38]:
# check unique values in each column
for col_name in messy.columns:
    unique_count = messy.select(col_name).distinct().show()
    print(f"{col_name}: {unique_count} unique values")

+-----------+
|employee_id|
+-----------+
|    EMP1097|
|    EMP1200|
|    EMP1214|
|    EMP1454|
|    EMP1896|
|    EMP2009|
|    EMP1351|
|    EMP1358|
|    EMP1452|
|    EMP1500|
|    EMP1653|
|    EMP1984|
|    EMP1075|
|    EMP1498|
|    EMP1581|
|    EMP1583|
|    EMP1839|
|    EMP1094|
|    EMP1491|
|    EMP1570|
+-----------+
only showing top 20 rows
employee_id: None unique values
+----------+
|First_Name|
+----------+
|     Grace|
|       Eva|
|     Heidi|
|   Charlie|
|       Bob|
|     Alice|
|     David|
|     Frank|
+----------+

First_Name: None unique values
+---------+
|Last_Name|
+---------+
|    Jones|
|    Davis|
| Williams|
|    Smith|
|   Miller|
|    Brown|
|   Garcia|
|  Johnson|
+---------+

Last_Name: None unique values
+----+
| Age|
+----+
|  40|
|  35|
|  25|
|  30|
|NULL|
+----+

Age: None unique values
+--------------------+
|   Department_Region|
+--------------------+
|     Finance-Florida|
|         HR-Illinois|
|      DevOps-Florida|
|         Admin-Te

In [42]:
categorical_cols = [
    c for c, t in messy.dtypes
    if t == "string"
]

for c in categorical_cols:
    print(f"\nValue counts for: {c}")

    messy.groupBy(c) \
        .count() \
        .orderBy(F.desc("count")) \
        .show(10, truncate=False)


Value counts for: employee_id


+-----------+-----+
|employee_id|count|
+-----------+-----+
|EMP1097    |1    |
|EMP1200    |1    |
|EMP1214    |1    |
|EMP1454    |1    |
|EMP1896    |1    |
|EMP2009    |1    |
|EMP1351    |1    |
|EMP1358    |1    |
|EMP1452    |1    |
|EMP1500    |1    |
+-----------+-----+
only showing top 10 rows

Value counts for: First_Name
+----------+-----+
|First_Name|count|
+----------+-----+
|Frank     |142  |
|Grace     |140  |
|Eva       |136  |
|Bob       |133  |
|Charlie   |125  |
|Alice     |117  |
|David     |116  |
|Heidi     |111  |
+----------+-----+


Value counts for: Last_Name
+---------+-----+
|Last_Name|count|
+---------+-----+
|Brown    |148  |
|Smith    |136  |
|Garcia   |136  |
|Jones    |131  |
|Davis    |120  |
|Johnson  |118  |
|Miller   |116  |
|Williams |115  |
+---------+-----+


Value counts for: Department_Region
+-----------------+-----+
|Department_Region|count|
+-----------------+-----+
|HR-Florida       |41   |
|DevOps-California|35   |
|Sales-Nevada     |35  

Lista to-do's:
- separar Department_Region C
- limpar notebook e meter markdowns R, C
- analisar duplicados mais a fundo R
- meter o "NULL" como missing value C
- corrigir datatypes (exemplo salario, date) R
- analisar melhor algumas colunas depois destas anomalias tratadas, exemplo, performance score é tudo string? C